# Análise dos Dados LiDAR

Exploração dos 3.152 tiles `.laz` do dataset **LiDAR Forest Inventory Brazil (ORNL DAAC, ID 1644)**.

**Fonte:** `data/raw/lidar/` — ver README dessa pasta para detalhes de download.

**Roteiro:**
1. Visão geral do dataset (tiles, sites, volume)
2. Distribuição por zona UTM
3. Distribuição de tamanho de arquivo e pontos por tile
4. Densidade de pontos estimada (pts/m²)
5. Cobertura espacial — mapa por site
6. Detalhe de um tile: estatísticas e visualização da nuvem de pontos

## 0. Setup

In [10]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import geopandas as gpd
import laspy

plt.rcParams.update({'figure.dpi': 130, 'font.size': 10})

ROOT = Path('.').resolve()
for _ in range(5):
    if (ROOT / 'data').exists(): break
    ROOT = ROOT.parent

LIDAR_DIR = next(p for p in (ROOT / 'data/raw/lidar').iterdir() if p.is_dir())
LIDAR_CSV = LIDAR_DIR / 'cms_brazil_lidar_tile_inventory.csv'

print('Dataset:', LIDAR_DIR.name)

Dataset: LiDAR_Forest_Inventory_Brazil_1644_1-20260505_011031


## 1. Visão geral

In [11]:
df = pd.read_csv(LIDAR_CSV)
print(f'Tiles totais:        {len(df):,}')
print(f'Sites únicos:        {df["filename"].str.split("_").str[0].nunique()}')
print(f'Zonas UTM:           {sorted(df["utmzone"].unique())}')
print(f'Volume total:        {df["file_size_mb"].sum()/1024:.1f} GB')
print(f'Versões LAS:         {df["version"].unique()}')
print(f'Formatos:            {df["file_format"].unique()}')
print()

Tiles totais:        3,152
Sites únicos:        50
Zonas UTM:           ['19S', '20S', '21S', '22S', '23S', '24S']
Volume total:        122.9 GB
Versões LAS:         [1.2 1.  1.1]
Formatos:            <StringArray>
['LAS/LAZ']
Length: 1, dtype: str



## 2. Resumo por site

In [12]:
df['site'] = df['filename'].str.split('_').str[0].str.upper()

by_site = df.groupby('site').agg(
    n_tiles=('filename', 'count'),
    total_gb=('file_size_mb', lambda x: x.sum() / 1024),
    utmzone=('utmzone', 'first'),
    lat_center=('min_lat', lambda x: (x.mean() + df.loc[x.index, 'max_lat'].mean()) / 2),
    lon_center=('min_lon', lambda x: (x.mean() + df.loc[x.index, 'max_lon'].mean()) / 2),
).reset_index().sort_values('n_tiles', ascending=False)

print(f'{len(by_site)} sites')
print(f'Maior site: {by_site.iloc[0]["site"]} ({by_site.iloc[0]["n_tiles"]} tiles)')
by_site

50 sites
Maior site: ST3 (419 tiles)


,site,n_tiles,total_gb,utmzone,lat_center,lon_center
45,ST3,419,6.353050,21S,-2.934133,-54.726041
38,PRG,391,3.797464,23S,-3.295026,-47.574704
26,FN3,262,3.569037,22S,-12.166924,-49.781848
21,CON,216,16.234453,24S,-14.496739,-39.103416
35,JAM,158,12.143286,20S,-9.194951,-62.979624
25,FN2,152,3.740893,21S,-11.859335,-54.195292
49,TAP,151,9.026546,21S,-3.062837,-54.961270
43,ST1,144,1.897130,21S,-3.505554,-54.937058
44,ST2,144,2.795054,21S,-3.488301,-54.841036
31,FST,140,7.532763,21S,-1.636682,-56.248545
